# 06 · Tabular Q-learning

**Goal:** learn the optimal action-values from sampled experience only, with no access to $P$ and $R$, and check the result against the exact answer from value iteration.

Q-learning is stochastic approximation of the Bellman optimality equation:

$$Q(s,a)\leftarrow Q(s,a)+\alpha\Big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\Big].$$

It is **off-policy**: we can behave $\varepsilon$-greedily while learning about the greedy policy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 4x4 gridworld, actions: 0=up 1=right 2=down 3=left, terminals in two opposite corners,
# reward -1 per step (so the optimal policy takes the shortest path to a corner).
N = 4
S = N * N
A = 4
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]
TERMINALS = {0, S - 1}

P = np.zeros((S, A, S))        # P[s, a, s'] transition probabilities
R = np.full((S, A), -1.0)      # R[s, a] expected reward
for s in range(S):
    if s in TERMINALS:
        P[s, :, s] = 1.0
        R[s, :] = 0.0
        continue
    r, c = divmod(s, N)
    for a, (dr, dc) in enumerate(MOVES):
        r2, c2 = r + dr, c + dc
        if not (0 <= r2 < N and 0 <= c2 < N):
            r2, c2 = r, c                      # bumping into a wall = stay put
        P[s, a, r2 * N + c2] = 1.0
assert np.allclose(P.sum(axis=2), 1.0)         # every row is a probability distribution

In [ ]:
gamma = 0.9

# ground truth from value iteration (the agent below never sees P or R)
V = np.zeros(S)
for _ in range(2000):
    V = (R + gamma * P @ V).max(axis=1)
Q_star = R + gamma * P @ V

def env_step(s, a, rng):
    # sample next state from P (here deterministic) and return the reward
    s2 = rng.choice(S, p=P[s, a])
    return s2, R[s, a]

nonterminal = np.array([s for s in range(S) if s not in TERMINALS])

In [ ]:
def q_learning(episodes=3000, alpha=0.5, eps=0.3, seed=0, max_steps=100):
    rng = np.random.default_rng(seed)
    Q = np.zeros((S, A))
    errs = []
    for ep in range(episodes):
        s = rng.choice(nonterminal)                # exploring starts
        for _ in range(max_steps):
            a = rng.integers(A) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r = env_step(s, a, rng)
            target = r + gamma * (0.0 if s2 in TERMINALS else Q[s2].max())
            Q[s, a] += alpha * (target - Q[s, a])
            s = s2
            if s in TERMINALS:
                break
        if ep % 50 == 0:
            errs.append(np.max(np.abs(Q[nonterminal] - Q_star[nonterminal])))
    return Q, errs

Q, errs = q_learning()
print("max |Q - Q*| over non-terminal states:", errs[-1].round(4))
assert errs[-1] < 0.05

# the greedy policy should be optimal: compare values, since ties allow several optimal actions
pi_q = Q.argmax(axis=1)
P_pi = P[np.arange(S), pi_q]; R_pi = R[np.arange(S), pi_q]
V_pi = np.linalg.solve(np.eye(S) - gamma * P_pi, R_pi)
assert np.allclose(V_pi, V, atol=1e-6)
print("greedy policy from Q-learning is optimal")

plt.semilogy(np.arange(len(errs)) * 50, errs)
plt.xlabel("episode"); plt.ylabel("max |Q - Q*|"); plt.title("Q-learning convergence"); plt.show()

## What to notice

* With deterministic transitions a constant $\alpha=0.5$ converges. With **stochastic** transitions you need $\alpha_t\to0$ with $\sum\alpha_t=\infty$, $\sum\alpha_t^2<\infty$ (Robbins–Monro conditions).
* Every $(s,a)$ pair must keep being visited. That is what exploring starts and $\varepsilon>0$ provide.
* The tabular method stores $|\mathcal S||\mathcal A|$ numbers. Replacing the table by a neural network $Q_\theta$ is the step to DQN (and brings instability issues: the "deadly triad").

## Exercises

1. Try $\alpha\in\{0.05,0.2,0.9\}$ and $\varepsilon\in\{0,0.05,0.5\}$. Plot the error curves and explain each.
2. Make transitions stochastic (slip probability 0.2). Show constant $\alpha$ now leaves noise, and a decaying $\alpha_t=1/(1+0.01\,n(s,a))$ fixes it.
3. Implement **SARSA** (on-policy: target uses the action actually taken next). Compare its policy to Q-learning's on a gridworld with a cliff.